# METEOR + Bootstrap CIs + Paired Significance Tests for Table 6

This notebook closes two gaps in the Med-LLaMa3 paper's Table 6:

1. **Point 1 — METEOR.** §IV-A-2 says: *"For open-ended tasks, we employed BLEU, ROUGE, and METEOR for text generation quality."* But Table 6 reports only ROUGE and BLEU. This notebook adds METEOR.
2. **Point 3 — statistical machinery.** Contribution (4) promises *"All benchmark comparisons are accompanied by McNemar's test p-values and bootstrap 95% confidence intervals."* Tables 4–5 have those; Table 6 doesn't. This notebook adds:
   - bootstrap 95% CIs on every per-model metric, and
   - paired Wilcoxon signed-rank tests of `Med-LLaMa3.1 8B` vs. each of the three baselines, with **Bonferroni correction** across the full comparison grid.
   - a paired-bootstrap p-value as a robustness check (non-parametric, no rank assumption).

Same evaluation slice as the paper: **100 Medical Meadow Flashcards samples**. Same four models:

| Slot | Model |
|------|-------|
| Ours | `Med-LLaMa3.1 8B` |
| Baseline 1 | `Llama3-Med42-8B` |
| Baseline 2 | `Llama-3.1-8B-Instruct` |
| Baseline 3 | `medgemma-4b-it` |

Inputs: per-model JSONs of `{id, question, reference, prediction}` for the same 100 samples used in `rouge-blue-evaluate.ipynb`. Re-running generation is *not* required — this notebook consumes already-saved predictions, which keeps the analysis fully reproducible without GPUs. A small mock-data fallback is included so the notebook runs end-to-end as a sanity check before plugging in real predictions.


## 1. Setup

In [ ]:
# Uncomment on first run.
# !pip install -q evaluate==0.4.2 nltk==3.8.1 rouge-score==0.1.2 sacrebleu scipy pandas numpy
# import nltk
# nltk.download('wordnet'); nltk.download('omw-1.4'); nltk.download('punkt')


In [ ]:
import json
import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy import stats

import evaluate
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Make sure NLTK assets METEOR needs are present.
for pkg in ("wordnet", "omw-1.4", "punkt"):
    try:
        nltk.data.find(pkg)
    except LookupError:
        nltk.download(pkg, quiet=True)

np.random.seed(42)


## 2. Configuration

In [ ]:
PREDICTIONS_DIR = Path("./predictions")
OUTPUT_DIR      = Path("./results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Map nice display names -> the JSON file your existing pipeline saved.
# Each JSON is a list of {"id", "question", "reference", "prediction"}.
MODELS = {
    "MedLLaMa3.1_8B"      : "predictions_medllama3_8b.json",     # ours
    "Llama3_Med42_8B"     : "predictions_med42_8b.json",
    "Llama3.1_8B_Instruct": "predictions_llama31_8b_instruct.json",
    "medgemma_4b_it"      : "predictions_medgemma_4b.json",
}

REFERENCE_MODEL = "MedLLaMa3.1_8B"   # the "ours" column in pairwise tests
N_BOOTSTRAP     = 1000               # matches the paper's bootstrap setup
ALPHA           = 0.05
RNG_SEED        = 42


## 3. Load predictions

**Expected format** (one JSON per model):

```json
[
  {"id": 0, "question": "...", "reference": "PTH-independent hypercalcemia, ...", "prediction": "In the context of ..."},
  {"id": 1, ...},
  ...
]
```

If any prediction file is missing, the next cell falls back to deterministic mock data so the notebook still runs end-to-end. **Replace the predictions JSONs with your real ones to get publishable numbers.**

In [ ]:
def _mock_predictions(n: int = 100, model_seed: int = 0) -> List[Dict]:
    """Deterministic mock predictions for offline sanity-checking."""
    rng = np.random.default_rng(1234 + model_seed)
    refs = [
        "PTH-independent hypercalcemia caused by cancer, granulomatous disease, or vitamin D intoxication.",
        "Low estradiol production leads to genitourinary syndrome of menopause.",
        "Low mobility and bulging of TM is suggestive of acute otitis media.",
        "Insulin resistance is a key feature of type 2 diabetes mellitus.",
        "Streptococcus pneumoniae is the most common cause of community-acquired pneumonia.",
    ]
    out = []
    for i in range(n):
        ref = refs[i % len(refs)]
        # Perturb the reference to simulate a model output of varying quality
        words = ref.split()
        keep = rng.uniform(0.55, 0.95)
        kept = [w for w in words if rng.uniform() < keep]
        # Occasionally append filler so length ratios look realistic
        if rng.uniform() < 0.4:
            kept = kept + ["This", "involves", "several", "underlying", "mechanisms", "."]
        pred = " ".join(kept) if kept else ref
        out.append({"id": i, "question": f"mock-q-{i}", "reference": ref, "prediction": pred})
    return out


def load_predictions(models: Dict[str, str], pred_dir: Path) -> Dict[str, List[Dict]]:
    """Load real JSONs where they exist; otherwise fall back to mock data with a warning."""
    data = {}
    for idx, (name, fname) in enumerate(models.items()):
        path = pred_dir / fname
        if path.exists():
            with open(path) as f:
                data[name] = json.load(f)
            print(f"[ok]   {name}: loaded {len(data[name])} samples from {path}")
        else:
            data[name] = _mock_predictions(n=100, model_seed=idx)
            print(f"[mock] {name}: file not found at {path} — using mock data")
    # Sanity: all models must cover the same N samples in the same order.
    sizes = {k: len(v) for k, v in data.items()}
    assert len(set(sizes.values())) == 1, f"prediction files have different sizes: {sizes}"
    ref0 = [d["reference"] for d in next(iter(data.values()))]
    for k, v in data.items():
        these = [d["reference"] for d in v]
        assert these == ref0, f"{k} has different reference order than the first model — re-export aligned"
    return data


predictions_data = load_predictions(MODELS, PREDICTIONS_DIR)
N = len(next(iter(predictions_data.values())))
print(f"\nN samples per model: {N}")


## 4. Per-sample metric computation

We compute per-sample scores rather than corpus-level only — bootstrap CIs and paired tests both need a per-sample distribution.

- **ROUGE-1/2/L/Lsum** — F1, with stemmer on (matches `rouge-score` defaults used by `rouge-blue-evaluate.ipynb`).
- **METEOR** — added here. WordNet-based, accounts for synonymy & stemming, which is what the paper claimed to use.
- **BLEU-1..4** — sentence-level with `SmoothingFunction.method1` (avoids the zero-on-missing-4-gram pathology that plagues sentence BLEU).

In [ ]:
# Lazy-load metrics once (evaluate caches internally).
_rouge  = evaluate.load("rouge")
_meteor = evaluate.load("meteor")
_smooth = SmoothingFunction().method1


def per_sample_rouge(pred: str, ref: str) -> Dict[str, float]:
    out = _rouge.compute(predictions=[pred], references=[ref], use_stemmer=True)
    return {k: float(out[k]) for k in ("rouge1", "rouge2", "rougeL", "rougeLsum")}


def per_sample_meteor(pred: str, ref: str) -> float:
    # `evaluate`'s METEOR wrapper expects predictions: List[str], references: List[str|List[str]]
    return float(_meteor.compute(predictions=[pred], references=[ref])["meteor"])


def per_sample_bleu(pred: str, ref: str) -> Dict[str, float]:
    pred_toks = pred.lower().split()
    ref_toks  = [ref.lower().split()]
    weights = {
        1: (1.0, 0, 0, 0),
        2: (0.5, 0.5, 0, 0),
        3: (1/3, 1/3, 1/3, 0),
        4: (0.25, 0.25, 0.25, 0.25),
    }
    return {
        f"bleu{n}": float(sentence_bleu(ref_toks, pred_toks, weights=weights[n], smoothing_function=_smooth))
        for n in (1, 2, 3, 4)
    }


def compute_per_sample(records: List[Dict]) -> pd.DataFrame:
    rows = []
    for r in records:
        pred, ref = r["prediction"], r["reference"]
        row = {"id": r["id"]}
        row.update(per_sample_rouge(pred, ref))
        row["meteor"] = per_sample_meteor(pred, ref)
        row.update(per_sample_bleu(pred, ref))
        rows.append(row)
    return pd.DataFrame(rows)


per_sample_dfs: Dict[str, pd.DataFrame] = {}
for name, recs in predictions_data.items():
    per_sample_dfs[name] = compute_per_sample(recs)
    print(f"{name}: {len(per_sample_dfs[name])} per-sample scores computed")

# Quick sanity peek
list(per_sample_dfs.values())[0].head()


## 5. Bootstrap 95% CIs (Point 3a)

Standard percentile bootstrap on the per-sample scores: resample with replacement, take the mean each time, take the 2.5/97.5 percentiles. Same `N_BOOTSTRAP=1000` the paper uses for its other tables.

In [ ]:
METRIC_COLS = ["rouge1", "rouge2", "rougeL", "rougeLsum", "meteor",
               "bleu1", "bleu2", "bleu3", "bleu4"]


def bootstrap_ci(values: np.ndarray, n_bootstrap: int = N_BOOTSTRAP,
                 alpha: float = ALPHA, seed: int = RNG_SEED) -> Tuple[float, float, float]:
    """Percentile bootstrap CI for the mean of `values`."""
    rng = np.random.default_rng(seed)
    n = len(values)
    boot = rng.choice(values, size=(n_bootstrap, n), replace=True).mean(axis=1)
    lo = float(np.percentile(boot, 100 * alpha / 2))
    hi = float(np.percentile(boot, 100 * (1 - alpha / 2)))
    return float(values.mean()), lo, hi


ci_rows = []
for model_name, df in per_sample_dfs.items():
    for metric in METRIC_COLS:
        mean, lo, hi = bootstrap_ci(df[metric].values)
        ci_rows.append({"model": model_name, "metric": metric,
                        "mean": mean, "ci_lo": lo, "ci_hi": hi})
ci_df = pd.DataFrame(ci_rows)
ci_df.head(10)


## 6. Paired tests: Med-LLaMa3.1 8B vs. each baseline (Point 3b)

The samples are paired (every model sees the same 100 flashcards), so we use the **Wilcoxon signed-rank test** as the primary test — same family the paper already uses for the MMLU-subset analysis. We add a paired bootstrap p-value as a non-parametric robustness check.

Comparisons performed: 3 baselines × 9 metrics = **27 paired tests**, so we apply **Bonferroni correction** at α = 0.05 ⇒ corrected threshold ≈ 0.00185.

In [ ]:
def paired_wilcoxon(a: np.ndarray, b: np.ndarray) -> Tuple[float, float]:
    """Two-sided Wilcoxon signed-rank on paired samples a, b. Handles all-zero diffs."""
    diffs = np.asarray(a) - np.asarray(b)
    if np.allclose(diffs, 0.0):
        return float("nan"), 1.0
    # `wilcox` zero-method drops zero-diff pairs, which is the conventional choice.
    res = stats.wilcoxon(a, b, alternative="two-sided", zero_method="wilcox")
    return float(res.statistic), float(res.pvalue)


def paired_bootstrap_pvalue(a: np.ndarray, b: np.ndarray,
                            n_bootstrap: int = N_BOOTSTRAP,
                            seed: int = RNG_SEED) -> Tuple[float, float]:
    """Two-sided paired bootstrap p-value for the mean difference."""
    rng = np.random.default_rng(seed)
    diffs = np.asarray(a) - np.asarray(b)
    observed = diffs.mean()
    centered = diffs - observed
    n = len(diffs)
    boot = rng.choice(centered, size=(n_bootstrap, n), replace=True).mean(axis=1)
    # +1/+1 smoothing avoids p=0 for clearly significant cases.
    p = (np.sum(np.abs(boot) >= abs(observed)) + 1) / (n_bootstrap + 1)
    return float(observed), float(p)


ref_df = per_sample_dfs[REFERENCE_MODEL]
test_rows = []
for model_name, df in per_sample_dfs.items():
    if model_name == REFERENCE_MODEL:
        continue
    for metric in METRIC_COLS:
        a = ref_df[metric].values
        b = df[metric].values
        w_stat, w_p = paired_wilcoxon(a, b)
        boot_diff, boot_p = paired_bootstrap_pvalue(a, b)
        test_rows.append({
            "vs_model"        : model_name,
            "metric"          : metric,
            "ours_mean"       : float(a.mean()),
            "other_mean"      : float(b.mean()),
            "delta"           : float(a.mean() - b.mean()),
            "wilcoxon_stat"   : w_stat,
            "wilcoxon_p"      : w_p,
            "boot_p"          : boot_p,
        })
test_df = pd.DataFrame(test_rows)

# Bonferroni across the full grid (3 models × 9 metrics).
n_comparisons = len(test_df)
bonf_thresh = ALPHA / n_comparisons
test_df["wilcoxon_p_bonf"]      = (test_df["wilcoxon_p"] * n_comparisons).clip(upper=1.0)
test_df["boot_p_bonf"]          = (test_df["boot_p"] * n_comparisons).clip(upper=1.0)
test_df["sig_uncorrected"]      = test_df["wilcoxon_p"] < ALPHA
test_df["sig_bonferroni"]       = test_df["wilcoxon_p_bonf"] < ALPHA

print(f"Number of comparisons: {n_comparisons}")
print(f"Bonferroni threshold:  {bonf_thresh:.5f}")
print(f"Significant (uncorrected): {test_df['sig_uncorrected'].sum()} / {n_comparisons}")
print(f"Significant (Bonferroni):  {test_df['sig_bonferroni'].sum()} / {n_comparisons}")
test_df.head(9)


## 7. Publication-ready output

Two CSVs to drop straight into the manuscript:

- `table6_extended_with_ci_and_meteor.csv` — Table 6 with METEOR added and `mean [lo, hi]` cells everywhere.
- `table6_pvalues_bonferroni.csv` — companion table of Bonferroni-corrected Wilcoxon p-values for *Ours vs. each baseline*.

In [ ]:
def fmt_ci(mean: float, lo: float, hi: float, places: int = 4) -> str:
    return f"{mean:.{places}f} [{lo:.{places}f}, {hi:.{places}f}]"


def stars(p: float) -> str:
    if np.isnan(p):  return "n.s."
    if p < 0.001:    return "***"
    if p < 0.01:     return "**"
    if p < 0.05:     return "*"
    return "n.s."


# --- Extended Table 6 ---
ci_view = ci_df.copy()
ci_view["cell"] = ci_view.apply(lambda r: fmt_ci(r["mean"], r["ci_lo"], r["ci_hi"]), axis=1)
table6 = ci_view.pivot(index="metric", columns="model", values="cell").reindex(METRIC_COLS)
# Order columns: ours first, then baselines.
col_order = [REFERENCE_MODEL] + [m for m in MODELS if m != REFERENCE_MODEL]
table6 = table6[col_order]

print("=== Extended Table 6 (mean [95% CI]) ===")
print(table6.to_string())
table6.to_csv(OUTPUT_DIR / "table6_extended_with_ci_and_meteor.csv")

# --- p-value companion table (Bonferroni corrected) ---
test_view = test_df.copy()
test_view["cell"] = test_view.apply(
    lambda r: f"{r['delta']:+.4f} (p={r['wilcoxon_p_bonf']:.3f} {stars(r['wilcoxon_p_bonf'])})",
    axis=1,
)
pvals = test_view.pivot(index="metric", columns="vs_model", values="cell").reindex(METRIC_COLS)
print("\n=== Wilcoxon paired test, Med-LLaMa3.1 8B vs. baseline (Bonferroni-corrected p) ===")
print(pvals.to_string())
pvals.to_csv(OUTPUT_DIR / "table6_pvalues_bonferroni.csv")

# Full per-comparison detail (also save raw Wilcoxon, raw bootstrap p, both corrected variants).
test_df.to_csv(OUTPUT_DIR / "per_comparison_test_results.csv", index=False)
ci_df.to_csv(OUTPUT_DIR / "per_model_metric_cis.csv", index=False)

# Also dump per-sample scores so reviewers can re-run any analysis.
for model_name, df in per_sample_dfs.items():
    df.to_csv(OUTPUT_DIR / f"per_sample_{model_name}.csv", index=False)

print(f"\nAll artefacts written to: {OUTPUT_DIR.resolve()}")


## 8. What to add to the manuscript

In the paper:

1. **Update Table 6.** Add a `METEOR` row. Replace bare numbers with `mean [95% CI]` cells from `table6_extended_with_ci_and_meteor.csv`.
2. **Add a small companion table or footnote** to Table 6 listing Bonferroni-corrected Wilcoxon p-values for `Med-LLaMa3.1 8B` vs. each baseline, taken from `table6_pvalues_bonferroni.csv`.
3. **Update §IV-A-2** to specify the test used: *"Per-sample ROUGE/BLEU/METEOR scores were compared with paired Wilcoxon signed-rank tests; bootstrap 95% confidence intervals (1,000 resamples) are reported alongside means; family-wise error across the 3 × 9 comparison grid is controlled with Bonferroni correction (α = 0.05)."*
4. **Update §V (Limitations)** if any of the Med-LLaMa3-vs-baseline differences fail to survive Bonferroni — those should be labelled n.s., consistent with the paper's existing convention for the MMLU subsets.
